In [51]:
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [53]:
# ==================== 评估指标 ====================

def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def evaluate(y_true, y_pred, prefix=""):
    mae_val = mean_absolute_error(y_true, y_pred)
    rmse_val = rmse(y_true, y_pred)
    mape_val = mape(y_true, y_pred)
    print(f"{prefix}MAE:  {mae_val:.4f}")
    print(f"{prefix}RMSE: {rmse_val:.4f}")
    print(f"{prefix}MAPE: {mape_val:.2f}%")
    return {'MAE': mae_val, 'RMSE': rmse_val, 'MAPE': mape_val}

In [54]:
# ==================== 时序块交叉验证 ====================

class TimeSeriesBlockCV:
    """
    时序块交叉验证
    验证集为连续块，训练集为剩余部分（最多两块）
    """
    
    def __init__(self, n_splits=5):
        self.n_splits = n_splits
    
    def split(self, X):
        n_samples = len(X)
        fold_size = n_samples // self.n_splits
        
        for i in range(self.n_splits):
            val_start = i * fold_size
            val_end = (i + 1) * fold_size if i < self.n_splits - 1 else n_samples
            
            val_idx = np.arange(val_start, val_end)
            train_idx = np.concatenate([
                np.arange(0, val_start),
                np.arange(val_end, n_samples)
            ])
            
            yield train_idx, val_idx
    
    def get_n_splits(self):
        return self.n_splits


In [57]:
# ==================== Optuna优化版本 ====================

def lightgbm_optuna_cv(X_train, y_train, X_test, y_test, cat_indices=None,
                        n_trials=30, cv_folds=5, random_state=42, use_gpu=False):
    """
    LightGBM Optuna + 时序块交叉验证 + 测试集评估
    
    参数:
    ------
    X_train, y_train : 训练数据
    X_test, y_test : 测试数据
    cat_indices : list
        类别特征索引，如 [8, 9, 10, 11, 12, 13, 14]
    n_trials : int
        优化迭代次数
    cv_folds : int
        交叉验证折数
    use_gpu : bool
        是否使用GPU
    """
    
    # 确保是numpy数组
    X_train = np.array(X_train)
    X_test = np.array(X_test)
    y_train = np.array(y_train)
    y_test = np.array(y_test)
    
    # 类别特征处理：转为整数
    cat_features = cat_indices if cat_indices else []
    if cat_features:
        X_train = X_train.copy()
        X_test = X_test.copy()
        for col in cat_features:
            X_train[:, col] = X_train[:, col].astype(int)
            X_test[:, col] = X_test[:, col].astype(int)
    
    print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")
    print(f"类别特征索引: {cat_features}")
    print(f"优化次数: {n_trials}, 时序CV: {cv_folds}折, GPU: {use_gpu}")
    print("=" * 60)
    
    # 时序块交叉验证
    tscv = TimeSeriesBlockCV(n_splits=cv_folds)
    
    # 打印CV划分信息
    print("时序块划分:")
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
        print(f"  Fold {fold+1}: train={len(train_idx)}, val={len(val_idx)} "
              f"[val范围: {val_idx[0]}-{val_idx[-1]}]")
    print("=" * 60)
    
    all_results = []
    
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 15, 127),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        }
        
        fold_metrics = {'MAE': [], 'RMSE': [], 'MAPE': []}
        
        for train_idx, val_idx in tscv.split(X_train):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]
            
            # 创建Dataset
            train_data = lgb.Dataset(
                X_tr, label=y_tr,
                categorical_feature=cat_features if cat_features else 'auto',
                free_raw_data=False
            )
            val_data = lgb.Dataset(
                X_val, label=y_val,
                reference=train_data,
                free_raw_data=False
            )
            
            lgb_params = {
                'objective': 'regression',
                'metric': 'mae',
                'boosting_type': 'gbdt',
                'device': 'gpu' if use_gpu else 'cpu',
                'verbose': -1,
                'seed': random_state,
                **{k: v for k, v in params.items() if k != 'n_estimators'}
            }
            
            model = lgb.train(
                lgb_params,
                train_data,
                num_boost_round=params['n_estimators'],
                valid_sets=[val_data],
                callbacks=[lgb.early_stopping(30, verbose=False)]
            )
            
            y_pred = model.predict(X_val)
            fold_metrics['MAE'].append(mean_absolute_error(y_val, y_pred))
            fold_metrics['RMSE'].append(rmse(y_val, y_pred))
            fold_metrics['MAPE'].append(mape(y_val, y_pred))
        
        result = {
            'trial': trial.number + 1,
            'params': params,
            'MAE_mean': np.mean(fold_metrics['MAE']),
            'MAE_std': np.std(fold_metrics['MAE']),
            'RMSE_mean': np.mean(fold_metrics['RMSE']),
            'RMSE_std': np.std(fold_metrics['RMSE']),
            'MAPE_mean': np.mean(fold_metrics['MAPE']),
            'MAPE_std': np.std(fold_metrics['MAPE']),
        }
        all_results.append(result)
        
        print(f"Trial {trial.number+1:3d}: MAE={result['MAE_mean']:.4f}, "
              f"RMSE={result['RMSE_mean']:.4f}, MAPE={result['MAPE_mean']:.2f}%")
        
        return result['MAE_mean']
    
    # 优化
    study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=random_state))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    best_params = study.best_params
    
    # 最终模型
    print("\n" + "=" * 60)
    print("使用最佳参数训练最终模型...")
    
    train_data_full = lgb.Dataset(
        X_train, label=y_train,
        categorical_feature=cat_features if cat_features else 'auto'
    )
    
    final_params = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'device': 'gpu' if use_gpu else 'cpu',
        'verbose': -1,
        'seed': random_state,
        **{k: v for k, v in best_params.items() if k != 'n_estimators'}
    }
    
    best_model = lgb.train(
        final_params,
        train_data_full,
        num_boost_round=best_params['n_estimators']
    )
    
    # 输出
    print("\n" + "=" * 60)
    print("最佳参数:")
    for k, v in best_params.items():
        print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")
    
    best_cv_result = min(all_results, key=lambda x: x['MAE_mean'])
    print(f"\n【交叉验证性能】")
    print(f"  MAE:  {best_cv_result['MAE_mean']:.4f} ± {best_cv_result['MAE_std']:.4f}")
    print(f"  RMSE: {best_cv_result['RMSE_mean']:.4f} ± {best_cv_result['RMSE_std']:.4f}")
    print(f"  MAPE: {best_cv_result['MAPE_mean']:.2f}% ± {best_cv_result['MAPE_std']:.2f}%")
    
    # 测试集评估
    print(f"\n【测试集性能】")
    y_pred_test = best_model.predict(X_test)
    test_metrics = evaluate(y_test, y_pred_test, prefix="  ")
    
    return best_model, best_params, all_results, test_metrics, y_pred_test

In [61]:
X_2d = np.load('./train_data_X_2d.npy')
y = np.load('./train_data_y.npy')

print("划分数据集...")
test_size = 24 * 365 * 1  # 最后2年作为测试集

X_train, X_test = X_2d[:-test_size], X_2d[-test_size:]
y_train, y_test = y[:-test_size], y[-test_size:]

划分数据集...


In [62]:
# Optuna版
best_model, best_params, cv_results, test_metrics, y_pred = lightgbm_optuna_cv(
    X_train, y_train, X_test, y_test,
    cat_indices=[8, 9, 10, 11, 12, 13, 14],
    n_trials=30,
    cv_folds=5,
    use_gpu=False
)

训练集: (69912, 21), 测试集: (8760, 21)
类别特征索引: [8, 9, 10, 11, 12, 13, 14]
优化次数: 30, 时序CV: 5折, GPU: False
时序块划分:
  Fold 1: train=55930, val=13982 [val范围: 0-13981]
  Fold 2: train=55930, val=13982 [val范围: 13982-27963]
  Fold 3: train=55930, val=13982 [val范围: 27964-41945]
  Fold 4: train=55930, val=13982 [val范围: 41946-55927]
  Fold 5: train=55928, val=13984 [val范围: 55928-69911]


  0%|          | 0/30 [00:00<?, ?it/s]

Trial   1: MAE=2219.5856, RMSE=3135.4499, MAPE=5.01%
Trial   2: MAE=2156.5230, RMSE=3054.7413, MAPE=4.86%
Trial   3: MAE=2162.2074, RMSE=3075.8828, MAPE=4.87%
Trial   4: MAE=2247.7540, RMSE=3147.4248, MAPE=5.09%
Trial   5: MAE=2190.3156, RMSE=3090.7932, MAPE=4.94%
Trial   6: MAE=2160.7418, RMSE=3080.2882, MAPE=4.87%
Trial   7: MAE=2218.8120, RMSE=3120.0464, MAPE=5.02%
Trial   8: MAE=2156.4531, RMSE=3068.6819, MAPE=4.86%
Trial   9: MAE=2217.0709, RMSE=3119.4247, MAPE=5.01%
Trial  10: MAE=2236.4207, RMSE=3139.0550, MAPE=5.07%
Trial  11: MAE=2145.8177, RMSE=3054.4343, MAPE=4.85%
Trial  12: MAE=2147.6110, RMSE=3056.5058, MAPE=4.85%
Trial  13: MAE=2147.0445, RMSE=3051.1581, MAPE=4.85%
Trial  14: MAE=2147.8540, RMSE=3052.8971, MAPE=4.85%
Trial  15: MAE=2166.6783, RMSE=3087.5950, MAPE=4.88%
Trial  16: MAE=2166.2374, RMSE=3078.4032, MAPE=4.88%
Trial  17: MAE=2159.2844, RMSE=3072.6754, MAPE=4.87%
Trial  18: MAE=2397.7826, RMSE=3297.3641, MAPE=5.45%
Trial  19: MAE=2156.2295, RMSE=3072.2242, MAPE

In [39]:
# # ==================== 快速网格搜索版本 ====================

# def lightgbm_fast_cv(X_train, y_train, X_test, y_test, cat_indices=None,
#                      cv_folds=5, random_state=42, use_gpu=False):
#     """快速网格搜索 + 时序块交叉验证"""
#     from itertools import product
    
#     # 确保是numpy数组
#     X_train = np.array(X_train)
#     X_test = np.array(X_test)
#     y_train = np.array(y_train)
#     y_test = np.array(y_test)
    
#     # 类别特征处理
#     cat_features = cat_indices if cat_indices else []
#     if cat_features:
#         X_train = X_train.copy()
#         X_test = X_test.copy()
#         for col in cat_features:
#             X_train[:, col] = X_train[:, col].astype(int)
#             X_test[:, col] = X_test[:, col].astype(int)
    
#     param_grid = {
#         'n_estimators': [200, 400, 600, 800, 1000],
#         'max_depth': [3, 4, 6, 8, 10],
#         'learning_rate': [0.02, 0.05, 0.1],
#         'num_leaves': [31, 63],
#     }
    
#     param_names = list(param_grid.keys())
#     param_combinations = list(product(*param_grid.values()))
#     total = len(param_combinations)
    
#     print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")
#     print(f"参数组合: {total}, 时序CV: {cv_folds}折")
#     print("=" * 60)
    
#     tscv = TimeSeriesBlockCV(n_splits=cv_folds)
    
#     # 打印划分
#     print("时序块划分:")
#     for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
#         print(f"  Fold {fold+1}: train={len(train_idx)}, val={len(val_idx)} "
#               f"[val范围: {val_idx[0]}-{val_idx[-1]}]")
#     print("=" * 60)
    
#     all_results = []
#     best_mae = float('inf')
#     best_params = None
    
#     for idx, params in enumerate(param_combinations):
#         current_params = dict(zip(param_names, params))
#         fold_metrics = {'MAE': [], 'RMSE': [], 'MAPE': []}
        
#         for train_idx, val_idx in tscv.split(X_train):
#             X_tr, X_val = X_train[train_idx], X_train[val_idx]
#             y_tr, y_val = y_train[train_idx], y_train[val_idx]
            
#             train_data = lgb.Dataset(
#                 X_tr, label=y_tr,
#                 categorical_feature=cat_features if cat_features else 'auto',
#                 free_raw_data=False
#             )
#             val_data = lgb.Dataset(
#                 X_val, label=y_val,
#                 reference=train_data,
#                 free_raw_data=False
#             )
            
#             lgb_params = {
#                 'objective': 'regression',
#                 'metric': 'mae',
#                 'boosting_type': 'gbdt',
#                 'device': 'gpu' if use_gpu else 'cpu',
#                 'verbose': -1,
#                 'seed': random_state,
#                 **{k: v for k, v in current_params.items() if k != 'n_estimators'}
#             }
            
#             model = lgb.train(
#                 lgb_params,
#                 train_data,
#                 num_boost_round=current_params['n_estimators'],
#                 valid_sets=[val_data],
#                 callbacks=[lgb.early_stopping(30, verbose=False)]
#             )
            
#             y_pred = model.predict(X_val)
#             fold_metrics['MAE'].append(mean_absolute_error(y_val, y_pred))
#             fold_metrics['RMSE'].append(rmse(y_val, y_pred))
#             fold_metrics['MAPE'].append(mape(y_val, y_pred))
        
#         result = {
#             'params': current_params,
#             'MAE_mean': np.mean(fold_metrics['MAE']),
#             'MAE_std': np.std(fold_metrics['MAE']),
#             'RMSE_mean': np.mean(fold_metrics['RMSE']),
#             'RMSE_std': np.std(fold_metrics['RMSE']),
#             'MAPE_mean': np.mean(fold_metrics['MAPE']),
#             'MAPE_std': np.std(fold_metrics['MAPE']),
#         }
#         all_results.append(result)
        
#         print(f"[{idx+1}/{total}] {current_params} → MAE={result['MAE_mean']:.4f}")
        
#         if result['MAE_mean'] < best_mae:
#             best_mae = result['MAE_mean']
#             best_params = current_params
    
#     # 最终模型
#     print("\n" + "=" * 60)
#     train_data_full = lgb.Dataset(
#         X_train, label=y_train,
#         categorical_feature=cat_features if cat_features else 'auto'
#     )
    
#     final_params = {
#         'objective': 'regression',
#         'metric': 'mae',
#         'boosting_type': 'gbdt',
#         'device': 'gpu' if use_gpu else 'cpu',
#         'verbose': -1,
#         'seed': random_state,
#         **{k: v for k, v in best_params.items() if k != 'n_estimators'}
#     }
    
#     best_model = lgb.train(
#         final_params,
#         train_data_full,
#         num_boost_round=best_params['n_estimators']
#     )
    
#     print(f"最佳参数: {best_params}")
    
#     best_cv_result = min(all_results, key=lambda x: x['MAE_mean'])
#     print(f"\n【交叉验证性能】")
#     print(f"  MAE:  {best_cv_result['MAE_mean']:.4f} ± {best_cv_result['MAE_std']:.4f}")
#     print(f"  RMSE: {best_cv_result['RMSE_mean']:.4f} ± {best_cv_result['RMSE_std']:.4f}")
#     print(f"  MAPE: {best_cv_result['MAPE_mean']:.2f}% ± {best_cv_result['MAPE_std']:.2f}%")
    
#     # 测试集
#     print(f"\n【测试集性能】")
#     y_pred_test = best_model.predict(X_test)
#     test_metrics = evaluate(y_test, y_pred_test, prefix="  ")
    
#     return best_model, best_params, all_results, test_metrics, y_pred_test

In [1]:
# # 快速网格搜索版
# best_model, best_params, cv_results, test_metrics, y_pred = lightgbm_fast_cv(
#     X_train, y_train, X_test, y_test,
#     cat_indices=[8, 9, 10, 11, 12, 13, 14],
#     cv_folds=5,
#     use_gpu=False
# )

In [47]:
# ==================== 保存模型 ====================

def save_model(model, best_params, test_metrics, model_name, save_dir='./models'):
    """
    保存模型、参数和指标
    
    参数:
        model: 训练好的模型
        best_params: 最佳参数字典
        test_metrics: 测试指标字典
        model_name: 模型名称 ('xgboost', 'lightgbm', 'catboost')
        save_dir: 保存目录
    """
    import os
    os.makedirs(save_dir, exist_ok=True)
    
    # 保存模型
    if model_name == 'xgboost':
        model.save_model(f'{save_dir}/{model_name}_model.json')
    elif model_name == 'lightgbm':
        model.save_model(f'{save_dir}/{model_name}_model.txt')
    elif model_name == 'catboost':
        model.save_model(f'{save_dir}/{model_name}_model.cbm')
    
    # 保存参数和指标
    info = {'best_params': best_params, 'test_metrics': test_metrics}
    with open(f'{save_dir}/{model_name}_info.json', 'w') as f:
        json.dump(info, f, indent=2)
    
    print(f"模型已保存到 {save_dir}/{model_name}_model.*")

In [48]:
# ==================== 读取模型 ====================

def load_model(model_name, save_dir='./models'):
    """
    读取模型、参数和指标
    
    返回:
        model, best_params, test_metrics
    """
    import xgboost as xgb
    import lightgbm as lgb
    from catboost import CatBoostRegressor
    
    # 读取模型
    if model_name == 'xgboost':
        model = xgb.Booster()
        model.load_model(f'{save_dir}/{model_name}_model.json')
    elif model_name == 'lightgbm':
        model = lgb.Booster(model_file=f'{save_dir}/{model_name}_model.txt')
    elif model_name == 'catboost':
        model = CatBoostRegressor()
        model.load_model(f'{save_dir}/{model_name}_model.cbm')
    
    # 读取参数和指标
    with open(f'{save_dir}/{model_name}_info.json', 'r') as f:
        info = json.load(f)
    
    print(f"模型已从 {save_dir}/{model_name}_model.* 加载")
    return model, info['best_params'], info['test_metrics']

In [63]:
save_model(best_model, best_params, test_metrics, "lightgbm")

模型已保存到 ./models/lightgbm_model.*


In [50]:
model, params, metrics = load_model('lightgbm')

模型已从 ./models/lightgbm_model.* 加载
